# PSDRL Transition Network + BLR Hyperparameter Search

Tunes the GRU transition model + Bayesian Linear Regression (BLR) on DeepSea 5×5 deterministic.

> **No autoencoder**: DeepSea uses 25-dim one-hot vector observations — already compact. `embed_dim = obs_dim = 25` (identity mapping). See `wiki/psdrl-autoencoder-analysis.md`.

**Two objectives:**
- **Obj 1 — `mean_grid_dist`**: Mean Euclidean distance (grid cells) between BLR-predicted and true next state, averaged over N_HYPOTHESES=20 BLR posterior samples.
- **Obj 2 — `env_steps_to_accuracy`**: Minimum environment transitions to achieve `mean_grid_dist < 0.5`. Probed at 250, 500; falls back to 1000.

**Search space:** `gru_dim`, `hidden_dim`, `learning_rate`, `window_length`, `transition_prior`, `reward_prior`

BLR priors correspond to `algorithm.transition_prior` and `algorithm.reward_prior` in `config.yaml`.

In [ ]:
import sys
sys.path.insert(0, '..')

import random, json, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
from optuna.samplers import NSGAIISampler
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

from bsuite.environments import deep_sea
from PSDRL.networks.transition import Network as TransitionNetwork
from PSDRL.common.settings import BLR_COEFFICIENT, ONE_OVER_LAMBDA

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

def get_device():
    if not torch.cuda.is_available():
        return 'cpu'
    try:
        torch.zeros(1).cuda(); torch.cuda.synchronize(); return 'cuda'
    except RuntimeError:
        return 'cpu'

DEVICE = get_device()
print(f'Device: {DEVICE}')

## Constants and Search Space

| Parameter | Choices | Description |
|---|---|---|
| `gru_dim` | [32, 64, 128, 256] | GRU hidden state size |
| `hidden_dim` | [64, 128, 256] | MLP layer width inside transition net |
| `learning_rate` | [1e-4, 3e-4, 1e-3] | Adam learning rate |
| `window_length` | [1, 2, 4] | TBPTT truncation window |
| `transition_prior` | [1e-4, 1e-3, 1e-2, 0.1] | BLR prior precision on state weights |
| `reward_prior` | [1e-4, 1e-3, 1e-2, 0.1] | BLR prior precision on reward weights |

In [ ]:
DS_SIZE         = 5
N_ACTIONS       = 2
OBS_DIM         = DS_SIZE * DS_SIZE   # 25 — raw obs = embedding (no AE)
N_GRID          = DS_SIZE
NOISE_VAR       = ONE_OVER_LAMBDA     # 1.0

GRU_DIM_CHOICES    = [32, 64, 128, 256]
HIDDEN_DIM_CHOICES = [64, 128, 256]
LR_CHOICES         = [1e-4, 3e-4, 1e-3]
WINDOW_CHOICES     = [1, 2, 4]
PRIOR_CHOICES      = [1e-4, 1e-3, 1e-2, 0.1]

TIME_LIMIT       = 1000
TEST_SIZE        = 200
PROBE_SIZES      = [250, 500]
TARGET_GRID_DIST = 0.5
N_HYPOTHESES     = 20   # BLR posterior samples per eval step

MAX_ITERS        = 200
PROBE_MAX_ITERS  = 100
PATIENCE         = 15
BATCH_SIZE       = 16  # episodes per gradient step

N_TRIALS         = 40

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

## Data Collection

Collect episodes from DeepSea 5×5 deterministic. Episodes are 5 steps long. Each step stores `(obs, action, next_obs, reward)`.

In [ ]:
def collect_episodes(n_transitions, ds_size=5, seed=42):
    """Collect episodes until at least n_transitions are gathered."""
    env = deep_sea.DeepSea(size=ds_size, deterministic=True, seed=seed)
    episodes = []
    total = 0

    while total < n_transitions:
        ts = env.reset()
        obs_list, act_list, next_list, rew_list = [], [], [], []

        while True:
            obs = torch.FloatTensor(ts.observation.flatten())
            action = np.random.randint(0, N_ACTIONS)
            ts = env.step(action)
            next_obs = torch.FloatTensor(ts.observation.flatten())

            obs_list.append(obs)
            act_list.append(action)
            next_list.append(next_obs)
            rew_list.append(float(ts.reward))
            total += 1

            if ts.last():
                break

        episodes.append({
            'obs':      torch.stack(obs_list),
            'actions':  torch.LongTensor(act_list),
            'next_obs': torch.stack(next_list),
            'rewards':  torch.FloatTensor(rew_list),
        })

    return episodes


def episodes_to_flat(episodes):
    """Flatten episodes into list of individual transitions."""
    trans = []
    for ep in episodes:
        T = len(ep['obs'])
        for t in range(T):
            trans.append((ep['obs'][t], ep['actions'][t], ep['next_obs'][t], ep['rewards'][t]))
    return trans


def episodes_from_n(episodes, n):
    """Return first n transitions worth of episodes."""
    result = []
    total = 0
    for ep in episodes:
        result.append(ep)
        total += len(ep['obs'])
        if total >= n:
            break
    return result


# Collect all data up front
all_episodes = collect_episodes(TIME_LIMIT + TEST_SIZE)

# Split train / test
cumlen = np.cumsum([len(ep['obs']) for ep in all_episodes])
train_cut = next(i for i, c in enumerate(cumlen) if c >= TIME_LIMIT)
train_episodes = all_episodes[:train_cut]
test_episodes  = all_episodes[train_cut:train_cut + max(1, TEST_SIZE // DS_SIZE)]

n_train = sum(len(ep['obs']) for ep in train_episodes)
n_test  = sum(len(ep['obs']) for ep in test_episodes)
print(f'Train: {len(train_episodes)} episodes, {n_train} transitions')
print(f'Test:  {len(test_episodes)} episodes, {n_test} transitions')

## GRU Training and BLR Evaluation Functions

In [ ]:
def build_episode_tensors(episodes, device):
    max_T = max(len(ep['obs']) for ep in episodes)
    N = len(episodes)
    obs_t = torch.zeros(N, max_T, OBS_DIM)
    act_t = torch.zeros(N, max_T, dtype=torch.long)
    next_t = torch.zeros(N, max_T, OBS_DIM)
    rew_t = torch.zeros(N, max_T, 1)
    lengths = [len(ep['obs']) for ep in episodes]
    for i, ep in enumerate(episodes):
        T = lengths[i]
        obs_t[i, :T] = ep['obs']; act_t[i, :T] = ep['actions']
        next_t[i, :T] = ep['next_obs']; rew_t[i, :T] = ep['rewards'].unsqueeze(-1)
    return obs_t.to(device), act_t.to(device), next_t.to(device), rew_t.to(device), lengths, max_T


def train_model(net, episodes, n_iters, window_length, lr,
               patience=PATIENCE, device=DEVICE):
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    loss_fn   = nn.MSELoss()
    obs_t, act_t, next_t, rew_t, lengths, max_T = build_episode_tensors(episodes, device)
    N = len(episodes)
    best, pat, e_conv = float('inf'), 0, n_iters
    for it in range(n_iters):
        idx = torch.randint(0, N, (BATCH_SIZE,))
        obs_b = obs_t[idx]; act_b = act_t[idx]
        next_b = next_t[idx]; rew_b = rew_t[idx]
        ep_lens = [lengths[i.item()] for i in idx]
        h = torch.zeros(BATCH_SIZE, net.gru_dim, device=device)
        it_loss, n_win = 0.0, 0
        for start in range(0, max_T, window_length):
            end = min(start + window_length, max_T)
            optimizer.zero_grad()
            w_loss = torch.tensor(0.0, device=device)
            for t in range(start, end):
                valid = [j for j in range(BATCH_SIZE) if t < ep_lens[j]]
                if not valid: break
                vi = torch.tensor(valid, device=device)
                act_oh = F.one_hot(act_b[vi, t], num_classes=N_ACTIONS).float()
                sa = torch.cat([obs_b[vi, t], act_oh], dim=-1)
                pred, h_new = net.forward(sa, h[valid])
                h[valid] = h_new
                w_loss = w_loss + loss_fn(pred[:, :-1], next_b[vi, t]) + loss_fn(pred[:, -1:], rew_b[vi, t])
            if w_loss.item() == 0.0: continue
            (w_loss / (end - start)).backward()
            optimizer.step(); h = h.detach()
            it_loss += w_loss.item() / (end - start); n_win += 1
        avg = it_loss / max(1, n_win)
        if avg < best - 1e-5: best, pat, e_conv = avg, 0, it + 1
        else:
            pat += 1
            if pat >= patience: break
    return best, e_conv


def compute_feature_maps(net, episodes, device=DEVICE):
    feats, tgts = [], []
    net.eval()
    with torch.no_grad():
        for ep in episodes:
            h = torch.zeros(1, net.gru_dim, device=device)
            for t in range(len(ep['obs'])):
                s = ep['obs'][t].unsqueeze(0).to(device)
                a = ep['actions'][t].long().item()
                s1 = ep['next_obs'][t].to(device)
                r  = ep['rewards'][t].to(device)
                act_oh = F.one_hot(torch.tensor([a], device=device), N_ACTIONS).float()
                sa = torch.cat([s, act_oh], dim=-1)
                h  = net._cell(sa, h)
                feat = net.layers[:-1](torch.cat([h, sa], dim=-1))
                feats.append(feat.squeeze(0))
                tgts.append(torch.cat([s1, r.unsqueeze(0)], dim=-1))
    net.train()
    return torch.stack(feats), torch.stack(tgts)


def fit_blr(feats, tgts, transition_prior, reward_prior, device=DEVICE):
    x, y = feats, tgts
    latent = x.shape[1]
    eye = torch.eye(latent, device=device)
    Phi_pre = x.T @ x * NOISE_VAR
    coeff = float(BLR_COEFFICIENT)
    while True:
        try:
            chol = torch.linalg.cholesky(Phi_pre.double() + eye.double() * coeff)
            Phi  = torch.cholesky_inverse(chol).float()
        except torch.linalg.LinAlgError:
            coeff *= 10; continue
        if Phi.isnan().any(): coeff *= 10; continue
        break
    t_cov = Phi * transition_prior
    r_cov = Phi * reward_prior
    mu = torch.zeros(OBS_DIM + 1, latent, device=device)
    for i in range(OBS_DIM + 1):
        mu[i] = (NOISE_VAR * Phi) @ x.T @ y[:, i]
    return mu, t_cov, r_cov


def sample_w(mu, t_cov, r_cov, device=DEVICE):
    rs = torch.randn_like(mu)
    w  = torch.empty_like(mu)
    w[:-1] = mu[:-1] + (rs[:-1] @ t_cov)
    w[-1]  = mu[-1]  + (rs[-1]  @ r_cov)
    return w


def eval_model_blr(net, mu, t_cov, r_cov, episodes, device=DEVICE):
    """Mean grid dist averaged over N_HYPOTHESES BLR samples."""
    net.eval()
    all_dists = []
    with torch.no_grad():
        for ep in episodes:
            h = torch.zeros(1, net.gru_dim, device=device)
            for t in range(len(ep['obs'])):
                s  = ep['obs'][t].unsqueeze(0).to(device)
                a  = ep['actions'][t].long().item()
                s1 = ep['next_obs'][t]
                act_oh = F.one_hot(torch.tensor([a], device=device), N_ACTIONS).float()
                sa = torch.cat([s, act_oh], dim=-1)
                h  = net._cell(sa, h)
                feat = net.layers[:-1](torch.cat([h, sa], dim=-1)).squeeze(0)
                true_idx = s1.argmax().item()
                tr, tc = divmod(true_idx, N_GRID)
                step_d = []
                for _ in range(N_HYPOTHESES):
                    w = sample_w(mu, t_cov, r_cov, device)
                    pred_idx = (w @ feat)[:-1].argmax().item()
                    pr, pc = divmod(pred_idx, N_GRID)
                    step_d.append(((pr-tr)**2 + (pc-tc)**2)**0.5)
                all_dists.append(float(np.mean(step_d)))
    net.train()
    return float(np.mean(all_dists)) if all_dists else float('inf')

## Optuna Objective

In [ ]:
def objective(trial):
    gru_dim    = trial.suggest_categorical('gru_dim',          GRU_DIM_CHOICES)
    hidden_dim = trial.suggest_categorical('hidden_dim',       HIDDEN_DIM_CHOICES)
    lr         = trial.suggest_categorical('learning_rate',    LR_CHOICES)
    window_len = trial.suggest_categorical('window_length',    WINDOW_CHOICES)
    tr_prior   = trial.suggest_categorical('transition_prior', PRIOR_CHOICES)
    rw_prior   = trial.suggest_categorical('reward_prior',     PRIOR_CHOICES)

    config = {'gru_dim': gru_dim, 'hidden_dim': hidden_dim, 'learning_rate': lr}

    # --- Objective 1: full dataset ---
    net = TransitionNetwork(OBS_DIM, N_ACTIONS, config, DEVICE)
    _, e_conv = train_model(net, train_episodes, MAX_ITERS, window_len, lr)
    feats, tgts = compute_feature_maps(net, train_episodes)
    mu, t_cov, r_cov = fit_blr(feats, tgts, tr_prior, rw_prior)
    mgd = eval_model_blr(net, mu, t_cov, r_cov, test_episodes)
    trial.set_user_attr('epochs_to_converge', e_conv)

    # --- Objective 2: env_steps_to_accuracy ---
    env_steps_to_accuracy = TIME_LIMIT
    for n in PROBE_SIZES:
        p_eps = episodes_from_n(train_episodes, n)
        p_net = TransitionNetwork(OBS_DIM, N_ACTIONS, config, DEVICE)
        train_model(p_net, p_eps, PROBE_MAX_ITERS, window_len, lr)
        p_feats, p_tgts = compute_feature_maps(p_net, p_eps)
        p_mu, p_tc, p_rc = fit_blr(p_feats, p_tgts, tr_prior, rw_prior)
        if eval_model_blr(p_net, p_mu, p_tc, p_rc, test_episodes) < TARGET_GRID_DIST:
            env_steps_to_accuracy = n
            break

    return mgd, float(env_steps_to_accuracy)

In [ ]:
sampler = NSGAIISampler(seed=42)
study = optuna.create_study(
    directions=['minimize', 'minimize'],
    sampler=sampler,
    study_name='psdrl_transition_blr',
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
print(f'\nCompleted {len(study.trials)} trials')

## Results Analysis

In [ ]:
def trials_to_df(study):
    rows = []
    for t in study.trials:
        if t.values is None: continue
        row = dict(t.params)
        row['mean_grid_dist']        = t.values[0]
        row['env_steps_to_accuracy'] = t.values[1]
        row['epochs_to_converge']    = t.user_attrs.get('epochs_to_converge', MAX_ITERS)
        row['pareto']                = t in study.best_trials
        rows.append(row)
    return pd.DataFrame(rows)


df = trials_to_df(study)
pareto_df = df[df['pareto']].sort_values('mean_grid_dist')

print(f'Total trials: {len(df)}')
print(f'Pareto-front trials: {len(pareto_df)}')
pareto_df[['gru_dim','hidden_dim','learning_rate','window_length',
           'transition_prior','reward_prior',
           'mean_grid_dist','env_steps_to_accuracy']].round(5)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(df['env_steps_to_accuracy'], df['mean_grid_dist'],
           c='lightsteelblue', alpha=0.5, s=30, label='All trials')
ax.scatter(pareto_df['env_steps_to_accuracy'], pareto_df['mean_grid_dist'],
           c='crimson', s=80, zorder=5, label='Pareto front')
ax.axhline(TARGET_GRID_DIST, color='orange', linestyle='--', alpha=0.7,
           label=f'Target ({TARGET_GRID_DIST})')
ax.set_xlabel('Env Steps to Accuracy (Obj 2)')
ax.set_ylabel('Mean Grid Distance (Obj 1)')
ax.set_title('PSDRL Transition+BLR — Pareto Front')
ax.legend(); ax.grid(True, alpha=0.3)

ax2 = axes[1]
for gd in sorted(df['gru_dim'].unique()):
    sub = df[df['gru_dim'] == gd]
    ax2.scatter(sub['env_steps_to_accuracy'], sub['mean_grid_dist'],
                alpha=0.6, s=40, label=f'gru_dim={gd}')
ax2.set_xlabel('Env Steps to Accuracy')
ax2.set_ylabel('Mean Grid Distance')
ax2.set_title('Coloured by gru_dim')
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## Best Model Evaluation

In [ ]:
if len(pareto_df) > 0:
    pdf = pareto_df.copy()
    for col in ['mean_grid_dist', 'env_steps_to_accuracy']:
        rng = pdf[col].max() - pdf[col].min()
        pdf[col + '_norm'] = (pdf[col] - pdf[col].min()) / (rng if rng > 0 else 1)
    pdf['score'] = pdf['mean_grid_dist_norm'] + pdf['env_steps_to_accuracy_norm']
    best_row = pdf.loc[pdf['score'].idxmin()]
else:
    best_row = df.loc[df['mean_grid_dist'].idxmin()]

best_config = {'gru_dim': int(best_row['gru_dim']),
               'hidden_dim': int(best_row['hidden_dim']),
               'learning_rate': float(best_row['learning_rate'])}
best_window = int(best_row['window_length'])
best_lr     = float(best_row['learning_rate'])
best_tr_p   = float(best_row['transition_prior'])
best_rw_p   = float(best_row['reward_prior'])

print('Best config:', best_config)
print(f'window_length={best_window}, transition_prior={best_tr_p}, reward_prior={best_rw_p}')
print(f'mean_grid_dist={best_row["mean_grid_dist"]:.4f}, env_steps={best_row["env_steps_to_accuracy"]:.0f}')

best_model = TransitionNetwork(OBS_DIM, N_ACTIONS, best_config, DEVICE)
final_loss, final_epochs = train_model(
    best_model, train_episodes, MAX_ITERS, best_window, best_lr)
best_feats, best_tgts = compute_feature_maps(best_model, train_episodes)
best_mu, best_tc, best_rc = fit_blr(best_feats, best_tgts, best_tr_p, best_rw_p)
final_mgd = eval_model_blr(best_model, best_mu, best_tc, best_rc, test_episodes)

n_params = sum(p.numel() for p in best_model.parameters())
print(f'Retrained → loss={final_loss:.6f}, test_mgd={final_mgd:.4f}')
print(f'Parameters: {n_params:,}  (BLR has no learned params)')

In [ ]:
best_model.eval()
state_dists = {s: [] for s in range(OBS_DIM)}

with torch.no_grad():
    for ep in test_episodes:
        h = torch.zeros(1, best_model.gru_dim, device=DEVICE)
        for t in range(len(ep['obs'])):
            s  = ep['obs'][t].unsqueeze(0).to(DEVICE)
            a  = ep['actions'][t].long().item()
            s1 = ep['next_obs'][t]
            state_idx = ep['obs'][t].argmax().item()
            act_oh = F.one_hot(torch.tensor([a], device=DEVICE), N_ACTIONS).float()
            sa = torch.cat([s, act_oh], dim=-1)
            h  = best_model._cell(sa, h)
            feat = best_model.layers[:-1](torch.cat([h, sa], dim=-1)).squeeze(0)
            true_idx = s1.argmax().item(); tr, tc = divmod(true_idx, N_GRID)
            step_d = []
            for _ in range(N_HYPOTHESES):
                w = sample_w(best_mu, best_tc, best_rc, DEVICE)
                pr, pc = divmod((w @ feat)[:-1].argmax().item(), N_GRID)
                step_d.append(((pr-tr)**2 + (pc-tc)**2)**0.5)
            state_dists[state_idx].append(float(np.mean(step_d)))

best_model.train()

state_means  = {s: np.mean(v) for s, v in state_dists.items() if v}
states_sorted = sorted(state_means.keys())
xtick_labels  = [f'{s}\n({s//N_GRID},{s%N_GRID})' for s in states_sorted]

fig, ax = plt.subplots(figsize=(max(10, len(states_sorted) * 0.5), 5))
ax.bar(range(len(states_sorted)), [state_means[s] for s in states_sorted],
       color='steelblue', alpha=0.8)
ax.axhline(TARGET_GRID_DIST, color='red', linestyle='--', alpha=0.7,
           label=f'Target ({TARGET_GRID_DIST} cells)')
ax.set_xticks(range(len(states_sorted)))
ax.set_xticklabels(xtick_labels, fontsize=8)
ax.set_xlabel('State index (row, col)')
ax.set_ylabel('Mean Grid Distance (BLR avg)')
ax.set_title('Per-State Prediction Accuracy — Best Transition+BLR Model')
ax.legend(); ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

## Save Optimal Parameters

In [ ]:
optimal_params = {
    'gru_dim':               int(best_row['gru_dim']),
    'hidden_dim':            int(best_row['hidden_dim']),
    'learning_rate':         float(best_row['learning_rate']),
    'window_length':         int(best_row['window_length']),
    'transition_prior':      float(best_row['transition_prior']),
    'reward_prior':          float(best_row['reward_prior']),
    'mean_grid_dist':        float(best_row['mean_grid_dist']),
    'env_steps_to_accuracy': int(best_row['env_steps_to_accuracy']),
    'epochs_to_converge':    int(best_row['epochs_to_converge']),
    'final_loss':            float(final_loss),
    'final_test_mgd':        float(final_mgd),
    'n_parameters':          n_params,
    'n_hypotheses_eval':     N_HYPOTHESES,
    'embed_dim':             OBS_DIM,
    'n_actions':             N_ACTIONS,
    'ds_size':               DS_SIZE,
    'n_trials':              N_TRIALS,
    'data_samples':          TIME_LIMIT,
}

out_path = Path('configs/psdrl_transition_params.json')
out_path.parent.mkdir(exist_ok=True)
with open(out_path, 'w') as f:
    json.dump(optimal_params, f, indent=2)
print(f'Saved to {out_path}')
print(json.dumps(optimal_params, indent=2))